<a href="https://colab.research.google.com/github/chaeyeonkimscientist/The-Body-Conducts-EEG/blob/main/notebooks/01_signal_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install mido for MIDI export
!pip install mido -q

import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt
import mido
from mido import MidiFile, MidiTrack, Message

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 1.9 MB/s eta 0:00:00


In [ ]:
# Uploading EEG data files

from google.colab import files
uploaded = files.upload()
# Upload: CapBaseline.txt, CapNback.txt, CapLinkedin.txt, CapBriefrecovery.txt

Saving CapBaseline.txt to CapBaseline.txt
Saving CapBriefrecovery.txt to CapBriefrecovery.txt
Saving CapLinkedin.txt to CapLinkedin.txt
Saving CapNback.txt to CapNback.txt


In [ ]:
# Preprocessing dataset

def load_and_clean(filename):
    df = pd.read_csv(filename, skiprows=4)
    df.columns = df.columns.str.strip()

    # Keep only the 4 EXG channels (no duplicate dropping on Sample Index!)
    eeg_cols = ['EXG Channel 0', 'EXG Channel 1', 'EXG Channel 2', 'EXG Channel 3']
    df = df[eeg_cols]

    # Convert to numeric and drop only rows with invalid data
    df = df.apply(pd.to_numeric, errors='coerce').dropna()

    return df.values  # shape: (n_samples, 4)

In [ ]:
# Alpha power function

FS           = 200
WIN_SEC      = 0.5 # Reduced from 2.0 to 0.5 seconds to accommodate smaller datasets
STEP_SEC     = 0.1 # Reduced from 0.5 to 0.1 seconds
WIN_SAMPLES  = int(WIN_SEC * FS)   # 100 samples (0.5 sec * 200 Hz)
STEP_SAMPLES = int(STEP_SEC * FS)  # 20 samples (0.1 sec * 200 Hz)

def bandpass_alpha(data, fs=FS, lowcut=8.0, highcut=13.0):
    nyq = fs / 2
    b, a = butter(4, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data, axis=0)

def compute_alpha_power(data):
    filtered = bandpass_alpha(data)
    n_samples = filtered.shape[0]
    powers = []
    start = 0
    while start + WIN_SAMPLES <= n_samples:
        window = filtered[start : start + WIN_SAMPLES]  # (100, 4)
        channel_powers = np.mean(window**2, axis=0)     # power per channel
        mean_power = np.mean(channel_powers)            # average across channels
        powers.append(np.log(mean_power + 1e-10))       # log
        start += STEP_SAMPLES
    return np.array(powers)

In [ ]:
# Computing baseline

baseline_data  = load_and_clean('CapBaseline.txt')
print(f"Shape of baseline_data after load_and_clean: {baseline_data.shape}")

# --- Debugging compute_alpha_power's input for baseline ---
filtered_baseline_data = bandpass_alpha(baseline_data)
print(f"Shape of filtered_baseline_data after bandpass_alpha: {filtered_baseline_data.shape}")
n_samples_for_alpha = filtered_baseline_data.shape[0]
print(f"Number of samples available for alpha power computation: {n_samples_for_alpha}")
print(f"Required WIN_SAMPLES for one window: {WIN_SAMPLES}")

if n_samples_for_alpha < WIN_SAMPLES:
    print(f"Warning: Not enough samples ({n_samples_for_alpha}) to create even one window ({WIN_SAMPLES} required) for alpha power computation. This will result in an empty baseline_alpha array and alpha_baseline being NaN.")
else:
    print("Sufficient samples for alpha power computation. Proceeding normally.")
# ---------------------------------------------------------

baseline_alpha = compute_alpha_power(baseline_data)
alpha_baseline = np.mean(baseline_alpha)

print(f"Baseline log alpha power: {alpha_baseline:.4f}")
print(f"Computed from {len(baseline_alpha)} windows ({len(baseline_alpha)*0.5:.1f} sec)")

Shape of baseline_data after load_and_clean: (16161, 4)
Shape of filtered_baseline_data after bandpass_alpha: (16161, 4)
Number of samples available for alpha power computation: 16161
Required WIN_SAMPLES for one window: 100
Sufficient samples for alpha power computation. Proceeding normally.
Baseline log alpha power: 10.9254
Computed from 804 windows (402.0 sec)


In [ ]:
# Stress signals, smoothing, normalization

def process_segment(filename, alpha_baseline, k=20):
    data  = load_and_clean(filename)
    alpha = compute_alpha_power(data)

    # Higher value = more alpha suppression = more stress
    stress_raw = alpha_baseline - alpha

    # Smooth (moving average, k=20 → 10 second window)
    stress_smooth = np.convolve(stress_raw, np.ones(k)/k, mode='same')

    # Normalize within this segment to 0–1
    s_min, s_max = stress_smooth.min(), stress_smooth.max()
    if s_max - s_min < 1e-10:
        stress_norm = np.zeros_like(stress_smooth)
    else:
        stress_norm = (stress_smooth - s_min) / (s_max - s_min)

    # Convert to MIDI integers 0–127
    stress_midi = np.round(stress_norm * 127).astype(int)
    return stress_midi

In [ ]:
# Runnin' time!

segments = {
    'CapNback':          'CapNback.txt',
    'CapBriefRecovery':  'CapBriefrecovery.txt',   # note lowercase 'r'
    'CapLinkedin':       'CapLinkedin.txt',
    'CapBaseline':       'CapBaseline.txt',
}

results = {}

# Check if alpha_baseline is a valid number before proceeding
if np.isnan(alpha_baseline):
    print("Error: Baseline alpha power could not be computed (alpha_baseline is NaN). Please ensure 'CapRecovery.txt' contains sufficient valid data and the 'compute_alpha_power' function can process it.")
else:
    for name, fname in segments.items():
        try:
            midi_vals = process_segment(fname, alpha_baseline, k=20)
            if midi_vals.size == 0:
                print(f"Warning: Skipping {name} because process_segment returned no valid MIDI values. Data for this segment might be insufficient or invalid.")
                results[name] = np.array([]) # Store an empty array for this segment
            else:
                results[name] = midi_vals
                print(f"{name}: {len(midi_vals)} windows ({len(midi_vals)*0.5:.1f} sec) | "
                      f"min={midi_vals.min()}  max={midi_vals.max()}  mean={midi_vals.mean():.1f}")
        except ValueError as e:
            print(f"Error processing {name}: {e}. This likely means insufficient or invalid data for the segment, causing an empty array to be passed to numpy.convolve.")
            results[name] = np.array([]) # Store an empty array for the failed segment

CapNback: 887 windows (443.5 sec) | min=0  max=127  mean=63.8
CapBriefRecovery: 551 windows (275.5 sec) | min=0  max=127  mean=50.3
CapLinkedin: 1068 windows (534.0 sec) | min=0  max=127  mean=52.8
CapBaseline: 804 windows (402.0 sec) | min=0  max=127  mean=35.5


In [ ]:
# Export as MIDI

def export_midi_cc(midi_values, filename, cc_number=11, tempo_bpm=60,
                   step_sec=0.5, channel=0):
    mid   = MidiFile(type=0)
    track = MidiTrack()
    mid.tracks.append(track)

    tempo          = mido.bpm2tempo(tempo_bpm)
    ticks_per_beat = mid.ticks_per_beat   # 480 default
    ticks_per_step = int((step_sec / (60 / tempo_bpm)) * ticks_per_beat)

    track.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))

    for i, val in enumerate(midi_values):
        time = ticks_per_step if i > 0 else 0
        track.append(Message('control_change',
                              channel=channel,
                              control=cc_number,
                              value=int(val),
                              time=time))
    mid.save(filename)
    print(f"Saved: {filename}  ({len(midi_values)} events)")

for name, midi_vals in results.items():
    export_midi_cc(midi_vals, f'{name}_stress.mid')

Saved: CapNback_stress.mid  (887 events)
Saved: CapBriefRecovery_stress.mid  (551 events)
Saved: CapLinkedin_stress.mid  (1068 events)
Saved: CapBaseline_stress.mid  (804 events)


In [ ]:
# Downloadinggg

for name in results.keys():
    files.download(f'{name}_stress.mid')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── AVERAGED MIDI: all four segments combined ──────────────────────────────
#
# Produces ONE stress MIDI file whose signal is the time-averaged
# stress curve across all four recordings:
#   CapBaseline, CapNback, CapLinkedin, CapBriefrecovery

import numpy as np
from scipy.interpolate import interp1d

def process_segment_float(filename, alpha_baseline, k=20):
    """Same as process_segment() but returns float 0-1 instead of int 0-127."""
    data  = load_and_clean(filename)
    alpha = compute_alpha_power(data)
    stress_raw    = alpha_baseline - alpha
    stress_smooth = np.convolve(stress_raw, np.ones(k)/k, mode='same')
    s_min, s_max  = stress_smooth.min(), stress_smooth.max()
    if s_max - s_min < 1e-10:
        return np.zeros_like(stress_smooth)
    return (stress_smooth - s_min) / (s_max - s_min)

segments_to_average = {
    'CapBaseline':      'CapBaseline.txt',
    'CapNback':         'CapNback.txt',
    'CapLinkedin':      'CapLinkedin.txt',
    'CapBriefRecovery': 'CapBriefrecovery.txt',
}

# Step 1: compute 0-1 float signals for each segment
float_signals = {}
for name, fname in segments_to_average.items():
    sig = process_segment_float(fname, alpha_baseline, k=20)
    float_signals[name] = sig
    print(f"{name}: {len(sig)} windows  ({len(sig)*STEP_SEC:.1f} sec)")

# Step 2: interpolate all to the length of the longest segment
max_len = max(len(s) for s in float_signals.values())

def interpolate_to_length(arr, target_len):
    if len(arr) == target_len:
        return arr
    x_old = np.linspace(0, 1, len(arr))
    x_new = np.linspace(0, 1, target_len)
    return interp1d(x_old, arr, kind='linear')(x_new)

resampled = {name: interpolate_to_length(sig, max_len)
             for name, sig in float_signals.items()}

# Step 3: element-wise average across all four
stacked  = np.stack(list(resampled.values()), axis=0)  # (4, max_len)
averaged = stacked.mean(axis=0)

# Step 4: re-normalise to 0-127
a_min, a_max = averaged.min(), averaged.max()
avg_norm = (averaged - a_min) / (a_max - a_min) if (a_max - a_min) > 1e-10 else np.zeros_like(averaged)
avg_midi = np.round(avg_norm * 127).astype(int)

# Step 5: stretch to song length accounting for tempo change
EXPORT_BPM = 60       # mido reference tempo

SECTION_1_SEC = 282   # 100 BPM section (0 to ~285s)
SECTION_2_SEC = 29    # 65 BPM section (285s to 311s)
BPM_1 = 80
BPM_2 = 65

# Events needed per section (scaled by BPM ratio so Logic plays them in correct wall-clock time)
events_sec1 = int(SECTION_1_SEC / STEP_SEC * (BPM_1 / EXPORT_BPM))
events_sec2 = int(SECTION_2_SEC / STEP_SEC * (BPM_2 / EXPORT_BPM))
total_events = events_sec1 + events_sec2

# Stretch avg_midi to total_events
x_old = np.linspace(0, 1, len(avg_midi))
x_new = np.linspace(0, 1, total_events)
avg_midi_stretched = np.round(interp1d(x_old, avg_midi.astype(float), kind='linear')(x_new)).astype(int)

# Split into two sections
chunk1 = avg_midi_stretched[:events_sec1]
chunk2 = avg_midi_stretched[events_sec1:]

# Build MIDI file manually with two tempo sections
mid   = mido.MidiFile(type=0)
track = mido.MidiTrack()
mid.tracks.append(track)

ticks_per_beat  = mid.ticks_per_beat  # 480
ticks_per_step1 = int((STEP_SEC / (60 / EXPORT_BPM)) * ticks_per_beat)
ticks_per_step2 = ticks_per_step1  # same step size, BPM difference handled by event count

track.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(EXPORT_BPM), time=0))

for i, val in enumerate(chunk1):
    track.append(Message('control_change', channel=0, control=11,
                         value=int(val), time=0 if i == 0 else ticks_per_step1))

# Tempo change message at the boundary
track.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(EXPORT_BPM), time=0))

for i, val in enumerate(chunk2):
    track.append(Message('control_change', channel=0, control=11,
                         value=int(val), time=ticks_per_step2))

mid.save('CapAveraged_stretched.mid')
print(f"Saved: chunk1={len(chunk1)} events, chunk2={len(chunk2)} events")
print(f"Expected duration: {len(chunk1)*STEP_SEC*(EXPORT_BPM/BPM_1) + len(chunk2)*STEP_SEC*(EXPORT_BPM/BPM_2):.1f} sec")
files.download('CapAveraged_stretched.mid')

CapBaseline: 804 windows  (80.4 sec)
CapNback: 887 windows  (88.7 sec)
CapLinkedin: 1068 windows  (106.8 sec)
CapBriefRecovery: 551 windows  (55.1 sec)
Saved: chunk1=3760 events, chunk2=314 events
Expected duration: 311.0 sec


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# SPECIFICALLY for Rot: Jazz piano lick conversion
# Mapping stress values (0-127) to pitches in a jazz scale
# and using stress intensity for velocity (how hard the note hits)

def export_jazz_piano_lick(midi_values, filename,
                           root_note=60,  # Middle C
                           scale_type='blues',
                           note_duration=0.25,  # quarter note
                           tempo_bpm=120):
    """
    Convert stress signal to jazz piano notes.

    Scale options:
    - 'blues': Blues scale (1, ♭3, 4, ♭5, 5, ♭7)
    - 'bebop': Bebop dominant (1, 2, 3, 4, 5, 6, ♭7, 7)
    - 'dorian': Dorian mode (1, 2, ♭3, 4, 5, 6, ♭7)
    - 'chromatic': All 12 notes
    """

    # Define jazz scales (intervals from root)
    scales = {
        'blues':     [0, 3, 5, 6, 7, 10],           # 6 notes
        'bebop':     [0, 2, 4, 5, 7, 9, 10, 11],    # 8 notes
        'dorian':    [0, 2, 3, 5, 7, 9, 10],        # 7 notes
        'chromatic': list(range(12))                # 12 notes
    }

    scale_intervals = scales[scale_type]

    # Map 0-127 stress values to 3 octaves of the scale
    def stress_to_pitch(stress_val):
        # Normalize to scale length across 3 octaves
        scale_length = len(scale_intervals)
        total_notes = scale_length * 3

        # Map 0-127 to note index
        note_idx = int((stress_val / 127.0) * (total_notes - 1))

        octave = note_idx // scale_length
        scale_degree = note_idx % scale_length

        pitch = root_note + (octave * 12) + scale_intervals[scale_degree]
        return min(max(pitch, 21), 108)  # Piano range: A0 to C8

         # --- Build list of (pitch, velocity) pairs ---
    note_list = []
    for stress_val in midi_values:
        pitch = stress_to_pitch(stress_val)
        velocity = int(60 + (stress_val / 127.0) * 60)
        note_list.append((pitch, velocity))

    # --- Merge consecutive notes with the same pitch ---
    merged = []
    i = 0
    while i < len(note_list):
        pitch, velocity = note_list[i]
        count = 1
        while i + count < len(note_list) and note_list[i + count][0] == pitch:
            count += 1
        merged.append((pitch, velocity, count))  # (pitch, velocity, duration_in_units)
        i += count


    mid = MidiFile(type=0)
    track = MidiTrack()
    mid.tracks.append(track)

    tempo = mido.bpm2tempo(tempo_bpm)
    track.append(mido.MetaMessage('set_tempo', tempo=tempo, time=0))
    track.append(mido.MetaMessage('track_name', name='Jazz Piano', time=0))

    # Piano patch (Acoustic Grand Piano)
    track.append(Message('program_change', program=0, time=0))

    ticks_per_beat = mid.ticks_per_beat
    ticks_per_note = int(note_duration * ticks_per_beat * (tempo_bpm / 60))

    # Create note sequence
    for i, stress_val in enumerate(midi_values):
        pitch = stress_to_pitch(stress_val)

        # Velocity: map stress to 60-120 (jazz dynamics)
        velocity = int(60 + (stress_val / 127.0) * 60)

        # Note on
        track.append(Message('note_on',
                            note=pitch,
                            velocity=velocity,
                            time=0 if i == 0 else ticks_per_note))

        # Note off (after duration)
        track.append(Message('note_off',
                            note=pitch,
                            velocity=0,
                            time=ticks_per_note))

    mid.save(filename)
    print(f"Saved jazz piano lick: {filename}")
    print(f"  {len(midi_values)} notes | Scale: {scale_type} | Root note: {root_note}")

In [ ]:
# Combine both stress experiences into one jazz lick
nback_data = results['CapNback']
linkedin_data = results['CapLinkedin']

# Interleave: nback, linkedin, nback, linkedin, nback...
min_length = min(len(results['CapNback']), len(results['CapLinkedin']))
interleaved = np.empty(min_length * 2, dtype=int)
interleaved[0::2] = results['CapNback'][:min_length]  # Even indices
interleaved[1::2] = results['CapLinkedin'][:min_length]  # Odd indices

export_jazz_piano_lick(
    interleaved,
    'Rot_jazz_lick_combined_mergednotes.mid',
    root_note=60,
    scale_type='bebop',  # Bebop might fit combined stress better
    note_duration=0.008,
    tempo_bpm=120
)

Saved jazz piano lick: Rot_jazz_lick_combined_mergednotes.mid
  1774 notes | Scale: bebop | Root note: 60


In [ ]:
# Download lick

files.download('Rot_jazz_lick_combined_mergednotes.mid')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>